# AI Stock Trading Agent — EDA and experiments

Walks through the pipeline step by step: data → EDA → features → LSTM → baselines → backtest.
Run it from the repository root (`jupyter lab`), so `src` is importable.

> Educational research only. Nothing here is financial advice.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():          # opened from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import PipelineConfig
from src.data_collection import load_prices
from src.eda import run_eda
from src.pipeline import run_pipeline
from src.preprocessing import build_features
from src.synthetic import generate_synthetic_ohlcv
from src.visualization import equity_chart, prediction_chart, training_chart

## 1. Load data
Set `USE_SYNTHETIC = True` only to run offline. Synthetic data is **artificial** and says nothing about real markets.
If the download fails, `load_prices` raises an error; it never substitutes other data.

In [ ]:
USE_SYNTHETIC = False
TICKER, START = "AAPL", "2015-01-01"

if USE_SYNTHETIC:
    df = generate_synthetic_ohlcv()
    report = {"source": "synthetic (artificial)"}
else:
    df, report = load_prices(TICKER, START, cache_dir=ROOT / "data" / "cache")
print(report)
df.tail()

## 2. Exploratory data analysis

In [ ]:
eda = run_eda(df)
for line in eda["findings"]:
    print("-", line)
eda["summary"].round(4)

In [ ]:
for name in ("price_ma", "returns", "volatility", "volume", "correlation"):
    display(eda["figures"][name]) if "display" in globals() else None
plt.show()

## 3. Features
Every feature at row *t* uses only data up to and including bar *t* (verified by unit tests that
truncate the history and compare rows).

In [ ]:
feats = build_features(df)
print(feats.shape, "features; rows lost to warm-up:", feats.isna().any(axis=1).sum())
feats.dropna().describe().T.round(4)

## 4. Train, compare with baselines, backtest
Target = next-bar simple return. Chronological 70/15/15 split with a purge gap, scalers fit on the training
window only, thresholds chosen from the validation period only. All headline numbers are on the untouched test split.

In [ ]:
cfg = PipelineConfig()
cfg.model.epochs = 40
result = run_pipeline(df, cfg)
print(result.split_summary, "\n")
print(result.comparison.round(5), "\n")
print(result.verdict)

In [ ]:
training_chart(result.history)

In [ ]:
prediction_chart(result.prediction_frame)

In [ ]:
print(result.threshold_info)
result.backtest.summary_table()

In [ ]:
equity_chart(result.backtest)

## 5. Reading the results honestly
* Compare the LSTM with the persistence, momentum and ridge baselines before drawing conclusions.
* A single test window is one sample of history; try other tickers and periods before trusting any number.
* Prediction accuracy is not profitability: check drawdown, costs and time in market.